In [0]:
BASE_PATH = "abfss://atininput@sadbtrng19032026.dfs.core.windows.net/data/"
STUDENT_ID = "u25"  # Change to your assigned u01–u16
DAY5 = BASE_PATH + "/day5"
P_BASIC = DAY5 + "/delta/flight_summary_basic"
DAY6_PREFIX = f"{BASE_PATH}day06-{STUDENT_ID}"

try:
    spark.read.format("delta").load(P_BASIC).limit(1).collect()
    print("✓ Day 5 tables found (P_BASIC)")
except Exception as e:
    print(f"✗ Day 5 tables not found: {e}")
    print("Please complete Day 5 notebook 01 first.")


# Delta performance: OPTIMIZE and ZORDER

Lab 1: write sample data, run OPTIMIZE with ZORDER, inspect the plan.  
Labs 2–5: partitioning, constraints, VACUUM, Bloom property — skip if you only need the minimum for the outline.

---

## Lab 1
### Step 1: Write Delta data


In [0]:
data = [(101, "C001", 5000, "2024-01-01"),
        (102, "C002", 7000, "2024-01-02"),
        (103, "C001", 4500, "2024-01-03"),
        (104, "C003", 3000, "2024-01-04"),
        (105, "C002", 8000, "2024-01-05")]

columns = ["transaction_id", "customer_id", "amount", "transaction_date"]

df = spark.createDataFrame(data, columns)
df.write.format("delta").mode("overwrite").save(f"{DAY6_PREFIX}/bank_transactions")


### Step 2: OPTIMIZE with ZORDER

In [0]:
spark.sql(f"OPTIMIZE delta.`{DAY6_PREFIX}/bank_transactions` ZORDER BY (customer_id)")


### Step 3: EXPLAIN a filter query

In [0]:
spark.read.format("delta").load(f"{DAY6_PREFIX}/bank_transactions").createOrReplaceTempView("bank_transactions")
spark.sql("EXPLAIN FORMATTED SELECT * FROM bank_transactions WHERE customer_id = 'C001'").show(truncate=False)


---

## Lab 2: Partitioned table

### Step 1: Create table


In [0]:
spark.sql("DROP TABLE IF EXISTS loan_foreclosures")
spark.sql(f"""
CREATE TABLE loan_foreclosures (
    loan_id STRING,
    customer_id STRING,
    outstanding_balance DOUBLE,
    foreclosure_date DATE
) USING DELTA
PARTITIONED BY (foreclosure_date)
LOCATION '{DAY6_PREFIX}/loan_foreclosures'
""")

### Step 2: Insert rows

In [0]:
from datetime import date

# Partition column is DATE; use date objects for insertInto
loan_data = [
    ("L001", "C001", 100000.0, date(2024, 1, 1)),
    ("L002", "C002", 150000.0, date(2024, 1, 2)),
    ("L003", "C001", 200000.0, date(2024, 1, 3)),
]
loan_columns = ["loan_id", "customer_id", "outstanding_balance", "foreclosure_date"]
loan_df = spark.createDataFrame(loan_data, loan_columns)


In [0]:
loan_df.write.insertInto("loan_foreclosures")


### Step 3: Query one partition

In [0]:
spark.sql("SELECT * FROM loan_foreclosures WHERE foreclosure_date = '2024-01-01'").show()


## Lab 3: CHECK constraint (optional)

### Step 1: Add constraint


In [0]:
try:
    spark.sql(f"ALTER TABLE delta.`{DAY6_PREFIX}/bank_transactions` ADD CONSTRAINT amt_nonneg CHECK (amount >= 0)")
except Exception as e:
    print("Constraint may already exist:", e)


## Lab 4: VACUUM (optional)

`VACUUM` deletes old files. The code relaxes the default retention check for a short demo; adjust or skip to match your policy.


In [0]:
# Optional: VACUUM after OPTIMIZE in Lab 1.
try:
    spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")
    spark.sql(f"VACUUM delta.`{DAY6_PREFIX}/bank_transactions` RETAIN 168 HOURS")
except Exception as e:
    print("VACUUM skipped or blocked:", e)


## Lab 5: Bloom filter property (optional)

Depends on DBR version.


In [0]:
try:
    spark.sql(f"ALTER TABLE delta.`{DAY6_PREFIX}/bank_transactions` SET TBLPROPERTIES ('delta.bloomFilter.columns' = 'customer_id')")
    print("Property set.")
except Exception as e:
    print("Skipped:", e)


## Wrap-up

Liquid clustering and data skipping fit the lecture and the product docs for your runtime.
